##### Building a collaborative-based book recommender system involves using the ratings given by users to recommend books to other users with similar preferences. We can use techniques like User-Based Collaborative Filtering or Item-Based Collaborative Filtering. For this guide, let's use User-Based Collaborative Filtering.

##### Let's review each dataset and decide if any actions are needed:


**Books Dataset:

**ISBN**: International Standard Book Number.
**Book-Title**: Title of the book.
**Book-Author**: Author of the book.
**Year-Of-Publication**: Year the book was published.
**Publisher**: Publisher of the book.
**Image-URL-S, Image-URL-M, Image-URL-L**: URLs for small, medium, and large images.

It seems like we don't need the image URLs for building the recommender system, but we will need this information later when we publish our application via streamlit. It will be enough to keep only one.

**Ratings Dataset:

**user_id**: User identifier.
**ISBN**: International Standard Book Number.
**rating**: User rating for the book.

It looks like there are no unnecessary columns here.

**Users Dataset:

**User-ID**: User identifier.
**Location**: Location of the user.
**Age**: Age of the user.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
books = pd.read_csv('books.csv', sep=";", error_bad_lines=False, encoding='latin-1')
ratings = pd.read_csv('ratings.csv', sep=";", error_bad_lines=False, encoding='latin-1')
users = pd.read_csv('users.csv', sep=";", error_bad_lines=False, encoding='latin-1')

Skipping line 6452: expected 8 fields, saw 9
Skipping line 43667: expected 8 fields, saw 10
Skipping line 51751: expected 8 fields, saw 9

Skipping line 92038: expected 8 fields, saw 9
Skipping line 104319: expected 8 fields, saw 9
Skipping line 121768: expected 8 fields, saw 9

Skipping line 144058: expected 8 fields, saw 9
Skipping line 150789: expected 8 fields, saw 9
Skipping line 157128: expected 8 fields, saw 9
Skipping line 180189: expected 8 fields, saw 9
Skipping line 185738: expected 8 fields, saw 9

Skipping line 209388: expected 8 fields, saw 9
Skipping line 220626: expected 8 fields, saw 9
Skipping line 227933: expected 8 fields, saw 11
Skipping line 228957: expected 8 fields, saw 10
Skipping line 245933: expected 8 fields, saw 9
Skipping line 251296: expected 8 fields, saw 9
Skipping line 259941: expected 8 fields, saw 9
Skipping line 261529: expected 8 fields, saw 9



In [3]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271360 non-null  object
 1   Book-Title           271360 non-null  object
 2   Book-Author          271359 non-null  object
 3   Year-Of-Publication  271360 non-null  object
 4   Publisher            271358 non-null  object
 5   Image-URL-S          271360 non-null  object
 6   Image-URL-M          271360 non-null  object
 7   Image-URL-L          271357 non-null  object
dtypes: object(8)
memory usage: 16.6+ MB


In [4]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1149780 entries, 0 to 1149779
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   User-ID      1149780 non-null  int64 
 1   ISBN         1149780 non-null  object
 2   Book-Rating  1149780 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 26.3+ MB


In [5]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278858 entries, 0 to 278857
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   User-ID   278858 non-null  int64  
 1   Location  278858 non-null  object 
 2   Age       168096 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 6.4+ MB


In [6]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [7]:
books.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='object')

In [8]:
books.drop(['Image-URL-S', 'Image-URL-M'], axis=1, inplace=True)

In [9]:
print(books.shape, users.shape, ratings.shape, sep='\n')

(271360, 6)
(278858, 3)
(1149780, 3)


In [10]:
books.rename(columns={'Book-Title':'title','Book-Author':'author','Year-Of-Publication':'year','Publisher':'publisher'},inplace=True)

In [11]:
# Drop rows with missing values in essential columns

books = books.dropna(subset=['author', 'publisher'])

In [12]:
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [13]:
users.rename(columns={'User-ID':'user_id','Location':'location','Age':'age'},inplace=True)

In [14]:
# Handle missing values in the Age column by filling with the median
# Convert Age to an integer type

users['age'] = users['age'].fillna(users['age'].median())
users['age'] = users['age'].astype(int)

In [15]:
# Filter out users below 18 or above 80

users = users[(users['age'] >= 18) & (users['age'] <= 80)]

In [16]:
ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [17]:
ratings.rename(columns={'User-ID':'user_id','Book-Rating':'rating'},inplace=True)

#### Improving the quality and reliability of the collaborative filtering system 

filtering out users with a low number of ratings is a common technique to improve the quality and reliability of collaborative filtering systems.  (ratings['user_id'].value_counts() > 200)

The code snippet we've provided is checking for users who have rated more than 200 books. This can be a helpful step in data preprocessing, and it depends on your specific goals and the characteristics of our dataset.

The idea behind such filtering is to focus on users who have provided a sufficient number of ratings. Users who have rated only a few books might not have a robust profile for collaborative filtering. By filtering out users with a low number of ratings, we could potentially improve the reliability of our recommender system.

After this step, our recommender system will focus on users who have rated more than 200 books. Setting the threshold (in this case, 200) depends on the characteristics of our dataset and the trade-off between having more data for collaborative filtering and focusing on users who are more active.

Keep in mind that this filtering step is optional, and we may want to experiment with different thresholds to see how it affects the performance of our recommender system. In addition, it's always a good idea to do some exploratory data analysis to understand the distribution of user ratings in your dataset before deciding on such thresholds.

Similarly, yo can set a threshold for the minimum **number of ratings per item**. This helps ensure that items with very few ratings don't overly influence recommendations.  
**Example**: Filter items with more than 50 ratings
popular_items = ratings['ISBN'].value_counts() > 50
popular_items = popular_items[popular_items].index.tolist()
ratings = ratings[ratings['ISBN'].isin(popular_items)]

**Normalize user ratings** to account for different rating scales or user biases. This can involve subtracting the user's mean rating from each of their ratings.
user_mean_ratings = ratings.groupby('user_id')['rating'].transform('mean')
ratings['normalized_rating'] = ratings['rating'] - user_mean_ratings 

Apply **weights to ratings** based on factors like recency or user activity. More recent ratings or ratings from more active users can be given higher weights.
( ratings['weighted_rating'] = ratings['rating'] * (1 + 0.1 * (current_year - ratings['timestamp'].dt.year)) )

If we want to do this based on **low-rated books**: ratings[ratings['rating'] >= 3] we will use this code.

In [18]:
active_users = ratings['user_id'].value_counts() > 200
active_users = active_users[active_users].index.tolist()

In [19]:
ratings = ratings[ratings['user_id'].isin(active_users)]

##### We combine ratings and book datasets to obtain the book titles associated with each rating.

In [20]:
ratings_with_books = ratings.merge(books, on='ISBN')

In [21]:
# Filter books by rating count

number_rating = ratings_with_books.groupby('title')['rating'].count().reset_index()

In [22]:
# Rename the column for clarity

number_rating.rename(columns={'rating':'num_of_rating'},inplace=True)

In [23]:
final_rating = ratings_with_books.merge(number_rating, on='title')
final_rating = final_rating[final_rating['num_of_rating'] >= 50]

In [24]:
final_rating.drop_duplicates(['user_id', 'title'], inplace=True)

##### Create User-Item Matrix

rows represent users, columns represent books, and entries represent user ratings. Fill NaN values with 0.

In [25]:
user_pivot_table = final_rating.pivot_table(columns='user_id', index='title', values='rating', fill_value=0)

Now that we have the user-item matrix, we can **compute the similarity** between users. One common method is to use the cosine similarity. We'll use the **cosine_similarity** function from scikit-learn

the dataset sizes provided, it's clear that the user-item matrix and the resulting cosine similarity matrix are quite large. This returns us an error.To address this, we'll use a sparse matrix representation and calculate cosine similarity incrementally.

In [31]:
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

user_item_matrix_array = user_pivot_table.T.to_numpy()

# Apply PCA to reduce dimensionality
pca = PCA(n_components=50)  # Adjust the number of components based on your memory constraints
user_item_matrix_pca = pca.fit_transform(user_item_matrix_array)

# Calculate cosine similarity on the reduced matrix
user_similarity = cosine_similarity(user_item_matrix_pca, dense_output=False)

# Convert the sparse cosine similarity matrix to a dense DataFrame
user_similarity_df = pd.DataFrame(user_similarity, index=user_pivot_table.columns, columns=user_pivot_table.columns)

The resulting user_similarity_df is a square matrix where each entry (i, j) represents the similarity between users i and j.

The purpose of creating this DataFrame is to have a structured representation of the cosine similarity values between users, which will be helpful for making recommendations.

This DataFrame will have user IDs as both index and columns, and the values will represent the cosine similarity between users. It will be used in the recommendation function to identify similar users for a given user.

#### Recommendation Function:

We will create a function that takes the user ID as input, calculates predicted ratings based on user similarity, and recommends books that the user has not yet rated.

the collaborative filtering function takes a user ID as input, calculates the predicted ratings for all items based on the user's similarity to other users, filters out items the user has already rated, and returns the top N recommendations based on predicted ratings

In [32]:
def get_user_recommendations(user_id, num_recommendations=5):
    # Extract the user's ratings from the user-item matrix
    user_ratings = user_item_matrix.loc[user_id].values.reshape(1, -1)
    
    # Calculate the cosine similarity between the user and all other users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:]
    
    # Weighted sum of ratings based on user similarity
    weighted_ratings = similar_users.values.reshape(-1, 1).T.dot(user_ratings)
    
    # Create a DataFrame with predicted ratings for all items
    recommendations = pd.DataFrame(weighted_ratings, columns=user_item_matrix.columns).T
    recommendations.columns = ['predicted_rating']
    
    # Filter out books the user has already rated
    user_books = ratings[ratings['user_id'] == user_id]['title']
    recommendations = recommendations[~recommendations.index.isin(user_books)]
    
    # Get top N recommendations based on predicted ratings
    top_recommendations = recommendations.sort_values(by='predicted_rating', ascending=False).head(num_recommendations)
    
    return top_recommendations

Now that the codes have been examined, let's see what we did step by step.

**Extract User Ratings**
This line retrieves the row corresponding to the given user_id from the user-item matrix. The ratings for each book are reshaped to a 2D array with one row and multiple columns.

**Calculate Cosine Similarity**
The function accesses the cosine similarity values from the user_similarity_df matrix for the given user and sorts them in descending order. The [1:] slice excludes the user's own similarity with themselves (which is always 1).

**Weighted Sum of Ratings**
This line computes the weighted sum of ratings based on the cosine similarity. It multiplies the similarity values by the user's ratings and sums them up to get a weighted sum.

**Create DataFrame with Predicted Ratings**
The function creates a DataFrame (recommendations) with the predicted ratings for all items. The transpose (T) operation is used to have items as rows and users as columns.

**Filter Out User's Rated Books**
This part filters out books that the user has already rated. It ensures that the recommender system does not recommend items the user has already interacted with.

**Get Top N Recommendations**
Finally, the function sorts the recommended items by predicted ratings in descending order and selects the top N items to recommend (num_recommendations).